In [1]:
import cv2
import numpy as np
import json
from pathlib import Path
from typing import Dict, Optional, Tuple, List

# Constants 

In [2]:
PITCH_W_M  = 105.0   # chiều dài sân (m) — trục X
PITCH_H_M  =  68.0   # chiều rộng sân (m) — trục Y
 
# Margin (px) xung quanh sân trên canvas 2D
MARGIN_LEFT   = 60
MARGIN_RIGHT  = 60
MARGIN_TOP    = 65
MARGIN_BOTTOM = 50
 
# Kích thước canvas output
CANVAS_W = 1050   # pixel
CANVAS_H =  720   # pixel
 
# Scale: số pixel trên 1 mét
SCALE_X = (CANVAS_W - MARGIN_LEFT - MARGIN_RIGHT)  / PITCH_W_M   # px/m
SCALE_Y = (CANVAS_H - MARGIN_TOP  - MARGIN_BOTTOM) / PITCH_H_M   # px/m
 
# Màu sắc
COLOR_GRASS_DARK  = (34,  85,  34)
COLOR_GRASS_LIGHT = (45, 100,  45)
COLOR_LINE        = (220, 220, 220)
COLOR_DOT_DEFAULT = (255, 255, 255)
COLOR_LABEL_BG    = (0,   0,   0)
COLOR_TEXT        = (255, 255, 255)
COLOR_TRAIL_ALPHA = 0.35
 
DOT_RADIUS  = 8     # px bán kính chấm tròn cầu thủ
TRAIL_LEN   = 30    # số frame lưu vết di chuyển

In [3]:
_PALETTE = [
    (255,  80,  80), (255, 165,   0), (255, 215,   0), (120, 230,  80),
    ( 50, 220, 120), (  0, 200, 180), ( 30, 160, 255), ( 80, 100, 255),
    (160,  80, 255), (255,  60, 180), (255, 140, 100), ( 80, 200, 200),
    (200, 100, 255), (255, 100, 140), (100, 200, 100), (255, 200,  50),
    ( 50, 180, 255), (255,  50, 100), (180, 255,  80), (100, 100, 255),
    (255, 180,  50), ( 80, 255, 160), (200,  80, 200), (255, 120,  50),
]
 
def _color(track_id: int) -> Tuple[int, int, int]:
    return _PALETTE[track_id % len(_PALETTE)]

In [4]:
def load_homography(keypoints_path: str) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load fisheye_keypoints.json và tính H (pixel→pitch), H_inv (pitch→pixel).
    """
    with open(keypoints_path, "r") as f:
        raw = json.load(f)
 
    src_pts, dst_pts = [], []
    for key, val in raw.items():
        x_pitch, y_pitch = eval(key)
        pixel_x, pixel_y = float(val[0]), float(val[1])
        src_pts.append([pixel_x, pixel_y])
        dst_pts.append([x_pitch, y_pitch])
 
    src_pts = np.float32(src_pts)
    dst_pts = np.float32(dst_pts)
 
    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 2.0)
    if H is None:
        raise RuntimeError("findHomography thất bại!")
 
    inliers = int(mask.ravel().sum())
    print(f"[Homography] Inliers: {inliers}/{len(src_pts)}")
 
    return H.astype(np.float64), np.linalg.inv(H).astype(np.float64)
 
 
def pixel_to_pitch(tlbr: List[float], H: np.ndarray) -> Optional[Tuple[float, float]]:
    """
    Bounding box [x1,y1,x2,y2] pixel → (x_m, y_m) tọa độ sân.
    Dùng điểm chân (bottom-center).
    """
    x1, y1, x2, y2 = tlbr
    bx = (x1 + x2) / 2.0
    by = float(y2)  # chân cầu thủ
 
    pt  = np.array([bx, by, 1.0], dtype=np.float64)
    dst = H @ pt
    if abs(dst[2]) < 1e-9:
        return None
    dst /= dst[2]
 
    x_m, y_m = dst[0], dst[1]
 
    # Lọc điểm nằm ngoài sân (thêm margin 5m)
    if not (-5 <= x_m <= PITCH_W_M + 5 and -5 <= y_m <= PITCH_H_M + 5):
        return None
 
    return float(x_m), float(y_m)

In [5]:
def pitch_to_canvas(x_m: float, y_m: float) -> Tuple[int, int]:
    """
    Tọa độ sân (m) → pixel trên canvas 2D.
    """
    cx = int(MARGIN_LEFT + x_m * SCALE_X)
    cy = int(MARGIN_TOP  + y_m * SCALE_Y)
    return cx, cy

def draw_pitch(canvas: np.ndarray) -> np.ndarray:
    """
    Vẽ sơ đồ sân bóng đá chuẩn lên canvas (kẻ đường, khu penalty, vòng tròn…).
    """
    # Nền cỏ xen kẽ sáng/tối (các dải dọc)
    n_stripes = 14
    stripe_w_m = PITCH_W_M / n_stripes
    for i in range(n_stripes):
        x0 = pitch_to_canvas(i * stripe_w_m, 0)[0]
        x1 = pitch_to_canvas((i + 1) * stripe_w_m, PITCH_H_M)[0]
        y0 = MARGIN_TOP
        y1 = CANVAS_H - MARGIN_BOTTOM
        color = COLOR_GRASS_DARK if i % 2 == 0 else COLOR_GRASS_LIGHT
        cv2.rectangle(canvas, (x0, y0), (x1, y1), color, -1)
 
    lc = COLOR_LINE  # line color
 
    def p(x, y):  # shorthand
        return pitch_to_canvas(x, y)
 
    # Đường biên sân
    cv2.rectangle(canvas, p(0, 0), p(PITCH_W_M, PITCH_H_M), lc, 2)
 
    # Đường giữa sân
    cv2.line(canvas, p(52.5, 0), p(52.5, PITCH_H_M), lc, 2)
 
    # Vòng tròn giữa sân (bán kính 9.15m)
    center = p(52.5, 34.0)
    r_px = int(9.15 * SCALE_X)
    cv2.circle(canvas, center, r_px, lc, 2)
    cv2.circle(canvas, center, 3, lc, -1)
 
    # Khu penalty trái (16.5m x 40.32m → y: 13.84~54.16)
    cv2.rectangle(canvas, p(0, 13.84), p(16.5, 54.16), lc, 2)
    # Khu cấm địa trái (5.5m x 18.32m → y: 24.84~43.16)
    cv2.rectangle(canvas, p(0, 24.84), p(5.5, 43.16), lc, 2)
 
    # Khu penalty phải
    cv2.rectangle(canvas, p(88.5, 13.84), p(PITCH_W_M, 54.16), lc, 2)
    # Khu cấm địa phải
    cv2.rectangle(canvas, p(99.5, 24.84), p(PITCH_W_M, 43.16), lc, 2)
 
    # Vòng cung penalty trái (bán kính 9.15m, tâm tại (11, 34))
    pen_l = p(11.0, 34.0)
    cv2.ellipse(canvas, pen_l, (int(9.15*SCALE_X), int(9.15*SCALE_Y)),
                0, -53, 53, lc, 2)
 
    # Vòng cung penalty phải (tâm tại (94, 34))
    pen_r = p(94.0, 34.0)
    cv2.ellipse(canvas, pen_r, (int(9.15*SCALE_X), int(9.15*SCALE_Y)),
                0, 127, 233, lc, 2)
 
    # Chấm penalty
    cv2.circle(canvas, p(11.0, 34.0), 4, lc, -1)
    cv2.circle(canvas, p(94.0, 34.0), 4, lc, -1)
 
    # Góc sân (cung ¼ vòng tròn, bán kính 1m)
    r_corner = int(1.0 * SCALE_X)
    for (cx_m, cy_m, ang_s, ang_e) in [
        (0,         0,         0,   90),
        (PITCH_W_M, 0,         90, 180),
        (PITCH_W_M, PITCH_H_M, 180, 270),
        (0,         PITCH_H_M, 270, 360),
    ]:
        cv2.ellipse(canvas, p(cx_m, cy_m), (r_corner, r_corner),
                    0, ang_s, ang_e, lc, 2)
 
    # Khung thành trái (y: 30.34~37.66)
    cv2.rectangle(canvas, p(0, 30.34), p(-2, 37.66), lc, 2)
 
    # Khung thành phải
    cv2.rectangle(canvas, p(PITCH_W_M, 30.34), p(PITCH_W_M + 2, 37.66), lc, 2)
 
    return canvas

In [6]:
def draw_players(
    canvas:    np.ndarray,
    positions: Dict[int, Tuple[float, float]],  # {tid: (x_m, y_m)}
    trails:    Dict[int, List[Tuple[int, int]]],  # {tid: [(cx,cy), ...]}
) -> np.ndarray:
    """
    Vẽ vết di chuyển (trail) và chấm tròn cầu thủ lên canvas.
    """
    overlay = canvas.copy()
 
    # Vẽ trail trước (dưới cùng)
    for tid, pts in trails.items():
        if len(pts) < 2:
            continue
        color = _color(tid)
        for i in range(1, len(pts)):
            alpha = i / len(pts)  # càng gần hiện tại càng đậm
            t_color = tuple(int(c * alpha) for c in color)
            thickness = max(1, int(2 * alpha))
            cv2.line(overlay, pts[i-1], pts[i], t_color, thickness, cv2.LINE_AA)
 
    # Blend trail với canvas gốc
    cv2.addWeighted(overlay, COLOR_TRAIL_ALPHA, canvas,
                    1 - COLOR_TRAIL_ALPHA, 0, canvas)
 
    # Vẽ chấm cầu thủ (trên cùng)
    for tid, (x_m, y_m) in positions.items():
        cx, cy = pitch_to_canvas(x_m, y_m)
        color = _color(tid)
 
        # Viền trắng + chấm màu
        cv2.circle(canvas, (cx, cy), DOT_RADIUS + 2, (255, 255, 255), -1, cv2.LINE_AA)
        cv2.circle(canvas, (cx, cy), DOT_RADIUS,     color,           -1, cv2.LINE_AA)
 
        # Label ID
        label = str(tid)
        font  = cv2.FONT_HERSHEY_SIMPLEX
        fs    = 0.42
        thick = 1
        (tw, th), _ = cv2.getTextSize(label, font, fs, thick)
 
        # Background label
        lx = cx - tw // 2 - 2
        ly = cy - DOT_RADIUS - th - 6
        cv2.rectangle(canvas, (lx, ly), (lx + tw + 4, ly + th + 4), color, -1)
        cv2.putText(canvas, label, (lx + 2, ly + th + 1),
                    font, fs, (255, 255, 255), thick, cv2.LINE_AA)
 
    return canvas

In [7]:
def _sanitize_pipeline_name(name: str) -> str:
    """
    Thay ký tự Unicode không hỗ trợ bởi OpenCV font thành ASCII tương đương.
    '→' / '->' đều được chuẩn hoá thành ' > '
    """
    return (name
            .replace("→", " > ")
            .replace("->", " > ")
            .replace("—", "-")
            .replace("–", "-"))
    


In [8]:
def _sanitize_pipeline_name(name: str) -> str:
    """
    Thay ký tự Unicode không hỗ trợ bởi OpenCV font thành ASCII tương đương.
    '→' / '->' đều được chuẩn hoá thành ' > '
    """
    return (name
            .replace("→", " > ")
            .replace("->", " > ")
            .replace("—", "-")
            .replace("–", "-"))

In [9]:
def draw_hud(
    canvas:     np.ndarray,
    frame_idx:  int,
    total:      int,
    n_players:  int,
    pipeline:   str,
) -> np.ndarray:
    """
    Vẽ thanh thông tin (HUD) 2 dòng phía trên canvas.
    Dòng 1: Pipeline name (ASCII-safe, tự wrap nếu quá dài)
    Dòng 2: Frame counter + Players + Progress bar
    """
    font       = cv2.FONT_HERSHEY_SIMPLEX
    pct        = frame_idx / max(total - 1, 1) * 100
    pipe_clean = _sanitize_pipeline_name(pipeline)
 
    # ── Tính layout: dòng pipeline có thể wrap ──
    fs_pipe  = 0.46
    fs_info  = 0.46
    thick    = 1
    pad      = 6          # padding ngang (px)
    line_h   = 22         # chiều cao mỗi dòng text (px)
    hud_h    = MARGIN_TOP # chiều cao tổng HUD = MARGIN_TOP
 
    # Nền HUD
    cv2.rectangle(canvas, (0, 0), (CANVAS_W, hud_h - 2), (10, 20, 10), -1)
 
    # ── Dòng 1: Pipeline (wrap thành nhiều đoạn nếu quá dài) ──
    max_w       = CANVAS_W - pad * 2
    words       = pipe_clean.split()
    lines       = []
    cur_line    = ""
    for word in words:
        test = (cur_line + " " + word).strip()
        (tw, _), _ = cv2.getTextSize(test, font, fs_pipe, thick)
        if tw <= max_w:
            cur_line = test
        else:
            if cur_line:
                lines.append(cur_line)
            cur_line = word
    if cur_line:
        lines.append(cur_line)
 
    for i, line in enumerate(lines):
        y = 16 + i * line_h
        cv2.putText(canvas, line, (pad, y),
                    font, fs_pipe, (160, 255, 120), thick, cv2.LINE_AA)
 
    # ── Dòng 2: Frame + Players (dòng cuối trước progress bar) ──
    info_y = hud_h - 12
 
    frame_txt = f"Frame {frame_idx + 1}/{total} ({pct:.1f}%)"
    cv2.putText(canvas, frame_txt,
                (pad, info_y), font, fs_info, (200, 200, 200), thick, cv2.LINE_AA)
 
    player_txt = f"Players: {n_players}"
    (ptw, _), _ = cv2.getTextSize(player_txt, font, fs_info, thick)
    cv2.putText(canvas, player_txt,
                (CANVAS_W - ptw - pad, info_y),
                font, fs_info, (255, 220, 80), thick, cv2.LINE_AA)
 
    # ── Progress bar (sát đáy HUD) ──
    bar_y = hud_h - 5
    bar_x0 = pad
    bar_w  = CANVAS_W - pad * 2
    cv2.rectangle(canvas, (bar_x0, bar_y), (bar_x0 + bar_w, bar_y + 4),
                  (50, 70, 50), -1)
    cv2.rectangle(canvas, (bar_x0, bar_y),
                  (bar_x0 + int(bar_w * pct / 100), bar_y + 4),
                  (80, 210, 80), -1)
 
    return canvas

In [10]:
def visualize_2d_pitch(
    all_tracks:     Dict,
    keypoints_path: str,
    output_path:    str = "output_2d_pitch.avi",
    fps:            float = 25.0,
    pipeline_name:  str = "YOLO26 → None → ByteTrack → GTALink(SOLIDER)",
    trail_len:      int  = TRAIL_LEN,
    show_preview:   bool = False,
) -> None:
    """
    Tạo video 2D hiển thị chuyển động cầu thủ trên sơ đồ sân.
 
    Parameters
    ----------
    all_tracks     : dict từ run_mot() / run_pipeline()['all_tracks']
                     {track_id: {'boxes': list[box|None], 'frames': list[int]}}
    keypoints_path : đường dẫn tới fisheye_keypoints.json
    output_path    : đường dẫn lưu video output (.avi)
    fps            : frame rate video output
    pipeline_name  : tên hiển thị trên HUD
    trail_len      : số frame lưu vết di chuyển
    show_preview   : hiển thị live preview (cần môi trường GUI)
    """
    # Load homography
    H, H_inv = load_homography(keypoints_path)
 
    # Xác định tổng số frame
    all_frame_indices = sorted(set(
        fi for data in all_tracks.values()
        for fi in data.get('frames', [])
    ))
    if not all_frame_indices:
        print("[ERROR] all_tracks rỗng — không có frame nào để visualize!")
        return
 
    max_frame = max(all_frame_indices)
    total_frames = max_frame + 1
 
    print(f"[2D Pitch] Tổng frames: {total_frames} | Tracks: {len(all_tracks)}")
    print(f"[2D Pitch] Output: {output_path}")
 
    # Xây frame_map: {frame_idx: {tid: [x1,y1,x2,y2]}}
    frame_map: Dict[int, Dict[int, List]] = {}
    for tid, data in all_tracks.items():
        boxes  = data.get('boxes', [])
        frames = data.get('frames', [])
        for fi in frames:
            if fi < len(boxes) and boxes[fi] is not None:
                frame_map.setdefault(fi, {})[tid] = boxes[fi]
 
    # Setup VideoWriter
    if not output_path.lower().endswith('.avi'):
        output_path = output_path.rsplit('.', 1)[0] + '.avi'
 
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (CANVAS_W, CANVAS_H))
    if not writer.isOpened():
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        writer = cv2.VideoWriter(output_path, fourcc, fps, (CANVAS_W, CANVAS_H))
 
    # Vẽ pitch template (tái sử dụng mỗi frame)
    pitch_template = np.zeros((CANVAS_H, CANVAS_W, 3), dtype=np.uint8)
    pitch_template = draw_pitch(pitch_template)
 
    # Trail buffer: {tid: deque of (cx, cy) canvas pixels}
    from collections import deque
    trails: Dict[int, deque] = {}
 
    # ── Render từng frame ──
    for frame_idx in range(total_frames):
        canvas = pitch_template.copy()
 
        boxes_this_frame = frame_map.get(frame_idx, {})
 
        # Tính tọa độ pitch cho từng cầu thủ
        positions: Dict[int, Tuple[float, float]] = {}
        for tid, tlbr in boxes_this_frame.items():
            xy_m = pixel_to_pitch(tlbr, H)
            if xy_m is None:
                continue
            x_m, y_m = xy_m
 
            # Clip vào trong sân
            x_m = np.clip(x_m, 0.0, PITCH_W_M)
            y_m = np.clip(y_m, 0.0, PITCH_H_M)
 
            positions[tid] = (x_m, y_m)
 
            # Cập nhật trail
            cx, cy = pitch_to_canvas(x_m, y_m)
            if tid not in trails:
                trails[tid] = deque(maxlen=trail_len)
            trails[tid].append((cx, cy))
 
        # Chỉ vẽ trail của những track còn active (hoặc mới mất <trail_len frame)
        active_trails = {tid: list(trails[tid]) for tid in positions if tid in trails}
 
        canvas = draw_players(canvas, positions, active_trails)
        canvas = draw_hud(canvas, frame_idx, total_frames,
                          len(positions), pipeline_name)
 
        writer.write(canvas)
 
        if show_preview:
            cv2.imshow("2D Pitch Visualization", canvas)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
 
        if frame_idx % 100 == 0:
            print(f"  Frame {frame_idx}/{total_frames} — {len(positions)} players")
 
    writer.release()
    if show_preview:
        cv2.destroyAllWindows()
 
    print(f"\n✅ Đã lưu video 2D: {output_path}")
    print(f"   → Mở bằng Windows Media Player hoặc VLC")
 

In [11]:
def add_2d_pitch_visualization(
    result:         Dict,
    keypoints_path: str,
    output_path:    str = "output_2d_pitch.avi",
    fps:            float = 25.0,
    pipeline_name:  str = "YOLO26 → None → ByteTrack → GTALink(SOLIDER)",
) -> None:
    """
    Wrapper gọi nhanh từ kết quả run_pipeline().
 
    Ví dụ:
        result_3 = run_pipeline(...)
        add_2d_pitch_visualization(
            result_3,
            keypoints_path=KP_PATH,
            output_path=os.path.join(OUTPUT_DIR, "2d_p03.avi"),
            pipeline_name="YOLO26 → None → ByteTrack → GTALink(SOLIDER)",
        )
    """
    all_tracks = result.get('all_tracks', result)  # hỗ trợ cả dict thuần
    visualize_2d_pitch(
        all_tracks     = all_tracks,
        keypoints_path = keypoints_path,
        output_path    = output_path,
        fps            = fps,
        pipeline_name  = pipeline_name,
    )

In [12]:
def _run_mot(video_path, detector, tracker, extractor=None,
             refiner=None, extractor_refine=None):
    """
    Chạy full MOT pipeline cho một video.
    Inline copy từ main.ipynb — dùng khi chạy visualize_2d_pitch.py độc lập.
 
    Returns
    -------
    all_tracks : dict
        {
            track_id (int): {
                'boxes' : list — box [x1,y1,x2,y2] hoặc None theo từng frame,
                'frames': list — frame_idx tương ứng (chỉ các frame có box),
                'feats' : list — embeddings (chỉ khi refiner != None)
            }
        }
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Không mở được video: {video_path}")
 
    # Dùng chung extractor nếu không có extractor_refine riêng
    if refiner is not None and extractor_refine is None:
        extractor_refine = extractor
 
    all_tracks = {}
    frame_idx  = 0
    frame_id   = 1
 
    while True:
        ret, frame = cap.read()
        if not ret:
            break
 
        print(f"Processing frame {frame_idx}", end="\r")
 
        # ── Detect ──
        detections = detector.detect(frame)
 
        # ── Crop + Feature ──
        valid_dets, crops = [], []
        for det in detections:
            x1, y1, x2, y2 = map(int, det[:-1])
            score = float(det[4])
            crop  = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            valid_dets.append((float(x1), float(y1), float(x2), float(y2), score))
            crops.append(crop)
 
        features = (extractor.extract_batch(crops)
                    if extractor is not None and crops
                    else [None] * len(crops))
 
        enriched_detections = [
            {'tlbr': [x1, y1, x2, y2], 'score': score, 'feat': feat}
            for (x1, y1, x2, y2, score), feat in zip(valid_dets, features)
        ]
 
        # ── Track ──
        active_tracks = tracker.update(enriched_detections, frame_id)
 
        # ── Lưu all_tracks ──
        active_ids = set()
        for track in active_tracks:
            tid = track.track_id
            active_ids.add(tid)
 
            if tid not in all_tracks:
                entry = {'boxes': [None] * frame_idx, 'frames': []}
                if refiner is not None:
                    entry['feats'] = []
                all_tracks[tid] = entry
 
            all_tracks[tid]['boxes'].append(track.tlbr.tolist())
            all_tracks[tid]['frames'].append(frame_idx)
 
        # ── Batch extract cho refiner ──
        if refiner is not None and active_tracks:
            refine_crops, refine_tids = [], []
            for track in active_tracks:
                x1, y1, x2, y2 = map(int, track.tlbr)
                crop = frame[y1:y2, x1:x2]
                refine_crops.append(crop if crop.size > 0 else frame[0:1, 0:1])
                refine_tids.append(track.track_id)
 
            refine_feats = extractor_refine.extract_batch(refine_crops)
            for tid, feat in zip(refine_tids, refine_feats):
                all_tracks[tid]['feats'].append(feat)
 
        # ── Track không active frame này → None ──
        for tid, data in all_tracks.items():
            if tid not in active_ids:
                if len(data['boxes']) == frame_idx:
                    data['boxes'].append(None)
 
        frame_idx += 1
        frame_id  += 1
 
    cap.release()
    print(f"\n[MOT] Done — {frame_idx} frames, {len(all_tracks)} tracks")
 
    if refiner is not None:
        return refiner.refine(all_tracks)
    return all_tracks

In [13]:
def _run_mot_baseline(video_path, detector, tracker, extractor,
                      frame_w, frame_h):
    """
    Chạy MOT pipeline cho Baseline: YOLOv5 → OSNet → DeepSort.
 
    Khác với _run_mot:
      - Không có refiner / GTALink
      - extract_batch trả về list (baseline FeatureExtractor trả về ndarray)
        nên cần normalize về list trước khi truyền vào tracker
 
    Parameters
    ----------
    video_path : str
    detector   : Detector (yolov5_hub hoặc yolov5)
    tracker    : Tracker  (deepsort)
    extractor  : FeatureExtractor (osnet)
    frame_w    : int — chiều rộng video (dùng cho DeepSort use_project)
    frame_h    : int — chiều cao video
 
    Returns
    -------
    all_tracks : dict {track_id: {'boxes': list[box|None], 'frames': list[int]}}
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Không mở được video: {video_path}")
 
    all_tracks = {}
    frame_idx  = 0
    frame_id   = 1
 
    while True:
        ret, frame = cap.read()
        if not ret:
            break
 
        print(f"[Baseline] Processing frame {frame_idx}", end="\r")
 
        # ── Detect ──
        detections = detector.detect(frame)
 
        # ── Crop + OSNet feature ──
        valid_dets, crops = [], []
        for det in detections:
            x1, y1, x2, y2 = map(int, det[:-1])
            score = float(det[4])
            crop  = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            valid_dets.append((float(x1), float(y1), float(x2), float(y2), score))
            crops.append(crop)
 
        # baseline extract_batch trả về ndarray (N,512) → chuyển thành list
        if crops:
            feats_arr = extractor.extract_batch(crops)   # (N,512) ndarray
            features  = [feats_arr[i] for i in range(len(feats_arr))]
        else:
            features = []
 
        enriched_detections = [
            {'tlbr': [x1, y1, x2, y2], 'score': score, 'feat': feat}
            for (x1, y1, x2, y2, score), feat in zip(valid_dets, features)
        ]
 
        # ── Track ──
        active_tracks = tracker.update(enriched_detections, frame_id)
 
        # ── Lưu all_tracks ──
        active_ids = set()
        for track in active_tracks:
            tid = track.track_id
            active_ids.add(tid)
 
            if tid not in all_tracks:
                all_tracks[tid] = {'boxes': [None] * frame_idx, 'frames': []}
 
            all_tracks[tid]['boxes'].append(track.tlbr.tolist())
            all_tracks[tid]['frames'].append(frame_idx)
 
        # ── Tracks không active → None ──
        for tid, data in all_tracks.items():
            if tid not in active_ids:
                if len(data['boxes']) == frame_idx:
                    data['boxes'].append(None)
 
        frame_idx += 1
        frame_id  += 1
 
    cap.release()
    print(f"\n[Baseline MOT] Done — {frame_idx} frames, {len(all_tracks)} tracks")
    return all_tracks

In [14]:
import os
import sys

# ════════════════════════════════════════════════════════════
# ĐƯỜNG DẪN — chỉnh sửa tại đây nếu cần
# ════════════════════════════════════════════════════════════
BASE          = r"D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT"
BASELINE_ROOT = r"D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\SoccerMOT_Baseline"

# PipelineMOT (YOLO26 + ByteTrack + GTALink)
YOLO26_PATH  = os.path.join(BASE, r"PipelineMOT\models\best_yolo26.pt")
SOLIDER_PATH = os.path.join(BASE, r"PipelineMOT\models\swin_small_converted.pth")
YOLO_PATH    = YOLO26_PATH   # alias để tương thích nếu cần

# Baseline (YOLOv5 + DeepSort + OSNet)
BEST_PT_PATH   = os.path.join(BASELINE_ROOT, r"src\models\best.pt")
YOLOV5M_PATH   = os.path.join(BASELINE_ROOT, r"src\models\yolov5m.pt")

# Dùng chung
VIDEO_PATH  = os.path.join(BASE, r"Data\wide_view\videos\F_20220220_1_1890_1920.mp4")
KP_PATH     = os.path.join(BASE, r"Data\fisheye_keypoints.json")
OUTPUT_DIR  = os.path.join(BASE, r"Output")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Đọc video info ──
_cap      = cv2.VideoCapture(VIDEO_PATH)
fps_video = _cap.get(cv2.CAP_PROP_FPS) or 25.0
FRAME_W   = int(_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
FRAME_H   = int(_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
_cap.release()
print(f"Video: {FRAME_W}x{FRAME_H} @ {fps_video:.1f}fps")

# ── Load Homography (dùng chung cho cả 2 pipeline) ──
H_mat, H_inv_mat = load_homography(KP_PATH)

Video: 6500x1000 @ 25.0fps
[Homography] Inliers: 62/65


In [15]:
print("\n" + "=" * 60)
print("PIPELINE 1: YOLO26 > ByteTrack > GTALink(SOLIDER)")
print("=" * 60)

sys.path.insert(0, os.path.join(BASE, "PipelineMOT"))
sys.path.insert(0, os.path.join(BASE, "PipelineMOT", "src"))

from src.feature_extractor import FeatureExtractor as FE_P1
from src.detector          import Detector          as Det_P1
from src.tracker           import Tracker           as Tracker_P1
from src.gtalink           import GTALink

detector_p1   = Det_P1(backend="yolo26", model_path=YOLO26_PATH)
extractor_p1  = FE_P1(
    backend="solider",
    solider_model_path=SOLIDER_PATH,
    solider_arch="swin_small",
    solider_semantic_weight=0.2,
)
tracker_p1 = Tracker_P1(algorithm="bytetrack", lambda_iou=1.0)
refiner_p1 = GTALink()

all_tracks_p1 = _run_mot(
    video_path       = VIDEO_PATH,
    detector         = detector_p1,
    tracker          = tracker_p1,
    extractor        = None,
    refiner          = refiner_p1,
    extractor_refine = extractor_p1,
)

visualize_2d_pitch(
    all_tracks     = all_tracks_p1,
    keypoints_path = KP_PATH,
    output_path    = os.path.join(OUTPUT_DIR, "best_pipeline.avi"),
    fps            = fps_video,
    pipeline_name  = "YOLO26 > ByteTrack > GTALink(SOLIDER)",
    trail_len      = 40,
)



PIPELINE 1: YOLO26 > ByteTrack > GTALink(SOLIDER)
[Detector] YOLO loaded on cuda
Missing: 0 | Unexpected: 8
[FeatureExtractor] SOLIDER (swin_small) loaded on cuda
[FeatureExtractor] Feature dim: 768
Processing frame 749
[MOT] Done — 750 frames, 55 tracks
[GTALink] Trước split: 55 tracklets


Splitting tracklets: 100%|██████████| 55/55 [00:00<00:00, 80.19it/s] 


[GTALink] Sau  split: 64 tracklets
[GTALink] Trước merge: 64 tracklets
[GTALink] Sau  merge: 27 tracklets
[Homography] Inliers: 62/65
[2D Pitch] Tổng frames: 750 | Tracks: 27
[2D Pitch] Output: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\Output\best_pipeline.avi
  Frame 0/750 — 20 players
  Frame 100/750 — 20 players
  Frame 200/750 — 21 players
  Frame 300/750 — 17 players
  Frame 400/750 — 21 players
  Frame 500/750 — 22 players
  Frame 600/750 — 18 players
  Frame 700/750 — 19 players

✅ Đã lưu video 2D: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\Output\best_pipeline.avi
   → Mở bằng Windows Media Player hoặc VLC


In [16]:
print("\n" + "=" * 60)
print("PIPELINE 2: YOLOv5m > OSNet > DeepSort")
print("=" * 60)

import importlib.util
import warnings
warnings.filterwarnings("ignore")

def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

# Xóa cache trước khi load Baseline — tránh xung đột tracker_alg
for key in list(sys.modules.keys()):
    if any(k in key for k in ("tracker", "deepsort", "tracker_alg", "src")):
        del sys.modules[key]

_bl = _load("bl_det", os.path.join(BASELINE_ROOT, "src", "detector.py"))
_fe = _load("bl_fe",  os.path.join(BASELINE_ROOT, "src", "feature_extractor.py"))
_tr = _load("bl_trk", os.path.join(BASELINE_ROOT, "src", "tracker.py"))

Det_P2      = _bl.Detector
FE_P2       = _fe.FeatureExtractor
Tracker_P2  = _tr.Tracker

detector_p2  = Det_P2(backend="yolov5_hub", model_path=YOLOV5M_PATH,
                      conf=0.4, iou=0.5, imgsz=1600)
extractor_p2 = FE_P2(backend="osnet", model_name="osnet_x1_0")
tracker_p2   = Tracker_P2(
    algorithm           = "deepsort",
    use_project         = True,
    H                   = H_mat,
    H_inv               = H_inv_mat,
    frame_w             = FRAME_W,
    frame_h             = FRAME_H,
    n_init              = 1,
    track_high_thresh   = 0.6,
    new_track_thresh    = 0.65,
    max_age             = 60,
    max_cosine_distance = 0.4,
    max_iou_distance    = 0.7,
)

all_tracks_p2 = _run_mot_baseline(
    video_path = VIDEO_PATH,
    detector   = detector_p2,
    tracker    = tracker_p2,
    extractor  = extractor_p2,
    frame_w    = FRAME_W,
    frame_h    = FRAME_H,
)

visualize_2d_pitch(
    all_tracks     = all_tracks_p2,
    keypoints_path = KP_PATH,
    output_path    = os.path.join(OUTPUT_DIR, "baseline.avi"),
    fps            = fps_video,
    pipeline_name  = "Baseline: YOLOv5m > OSNet > DeepSort",
    trail_len      = 40,
)


PIPELINE 2: YOLOv5m > OSNet > DeepSort


YOLOv5  2026-5-14 Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)

Fusing layers... 
YOLOv5m summary: 290 layers, 21172173 parameters, 0 gradients, 48.9 GFLOPs
Adding AutoShape... 


[Detector] YOLOv5-hub loaded | model=D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\SoccerMOT_Baseline\src\models\yolov5m.pt | device=cuda | conf=0.4 iou=0.5 imgsz=1600
[FeatureExtractor] Loading osnet_x1_0 pretrained ...
Successfully loaded imagenet pretrained weights from "C:\Users\Admin/.cache\torch\checkpoints\osnet_x1_0_imagenet.pth"
[FeatureExtractor] Ready on cuda
[Tracker] deepsort initialized | use_project=True
[Baseline] Processing frame 749
[Baseline MOT] Done — 750 frames, 55 tracks
[Homography] Inliers: 62/65
[2D Pitch] Tổng frames: 750 | Tracks: 55
[2D Pitch] Output: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\Output\baseline.avi
  Frame 0/750 — 5 players
  Frame 100/750 — 5 players
  Frame 200/750 — 12 players
  Frame 300/750 — 8 players
  Frame 400/750 — 11 players
  Frame 500/750 — 9 players
  Frame 600/750 — 7 players
  Frame 700/750 — 6 players

✅ Đã lưu video 2D: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_M